In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%load_ext autoreload
%autoreload 2

# Secondary pharmacology

This notebook shows you how to score ligands against the secondary-pharmacology
kinase panel using `SecondaryPharmacology`. There are two mutually-exclusive
ways to do this, picked with the required `method` argument:

- `method="ligand-ml"` — fast ML activity predictions, returned immediately.
- `method="docking"` — physically docks each ligand against the panel and
  scores the poses. Runs as a background job.

`method` has no default — the two paths differ enough in cost and latency
that picking one should be deliberate. `run()`, `start()`, and `watch()` all
exist on every instance, but only the one matching `method` works — the
others raise immediately, telling you which to call instead.

## Setup

In [ ]:
from deeporigin import projects
from deeporigin.drug_discovery import (
    SecondaryPharmacology,
    Ligand,
)
from deeporigin.platform import DeepOriginClient

`DeepOriginClient()` picks up your environment (dev/staging/prod) from your
local `deeporigin` config — no env argument needed here, same as every other
notebook in this repo.

In [ ]:
client = DeepOriginClient()
client

Select which project new runs and synced ligands land in — everything below
is scoped to whatever project is current, so make this explicit rather than
relying on whatever was last loaded:

In [ ]:
projects.load("test-asaglam")
projects.current()

## The panel

See what's currently in the panel. `panel()` previews the first 10 members by
default, with a count hint — pass `full=True` for the complete table:

In [ ]:
SecondaryPharmacology.panel()

In [ ]:
SecondaryPharmacology.panel(full=True)

## Create a ligand

Both paths take a ligand straight from SMILES — neither needs a local 3D
conformer or a protonation step first. Ligand-ML only ever fingerprints the
2D graph, and docking's payload here is `{id, smiles}` only (no coordinates),
so the engine generates its own conformer and protonation state server-side.
Give `name=` something readable; it's used to label plots later.

In [ ]:
ligand = Ligand.from_smiles(
    "COc1cc2ncnc(Nc3ccc(F)c(Cl)c3)c2cc1OCCCN1CCOCC1", name="gefitinib"
)
ligand

## Ligand-ML scoring

Fast, synchronous — `run()` returns a `DataFrame` directly, no `start()`/`wait()` needed.

In [ ]:
job = SecondaryPharmacology(ligands=[ligand], method="ligand-ml")
job

In [ ]:
df = job.run()
df

`plot()` is method-aware and renders straight from the run: for ligand-ml, a
heatmap of ligand × target colored by score. Rows are labeled by
`Ligand.name`, falling back to a short id suffix if unset -- disambiguated
either way.

In [ ]:
job.plot()

`df` has one row per ligand × panel member, with `uniprot_id`, `gene_name`,
and exactly one of `p_active` (classification) or `p_affinity` (regression,
-log10 M) set per row. Every current panel member is classification-only,
so `p_active` is always the one that's set.

`ligand_id` is a real platform id, even though `ligand` itself was never
explicitly synced — `run()` registers it after scoring (a no-op if it's
already registered) so results are always referable, not stuck at `None`.

Restrict to a subset of the panel with `uniprots`:

In [ ]:
job_egfr_only = SecondaryPharmacology(
    ligands=[ligand],
    method="ligand-ml",
    uniprots=["P00533"],  # EGFR only
)
job_egfr_only.run()

## Docking

Submitted as a background job — `start()`, then `wait()` (or `await watch()`
in a notebook for live progress) before pulling results.

In [ ]:
dock_job = SecondaryPharmacology(
    ligands=[ligand],
    method="docking",
    uniprots=["P00533"],  # EGFR only, keep this quick
    effort=1,
)
dock_job

In [ ]:
dock_job.start()
dock_job.wait()  # or: await dock_job.watch()

Allow a moment after `wait()`/`watch()` completes — results land in the
platform's data index rather than the immediate response.

In [ ]:
df = dock_job.get_results()
df

For docking, `plot()` defaults to a heatmap of ligand × target colored by
`binding_energy` -- same visual grammar as ligand-ml's. Pass
`metric="pose_score"` for the other one. (No combined pose_score-vs-
binding_energy view: the two are on unrelated scales, kcal/mol vs a ~0-1
score, so plotting one against the other would need real normalization to
mean anything.)

In [ ]:
dock_job.plot()

In [ ]:
dock_job.plot(metric="pose_score")

`df` has one row per docked pose, including `pose_score`, `binding_energy`,
`pdb_id`, and `file_path` -- `pose_score`/`binding_energy` are the way to
consume docking results today. Pose *visualization* isn't available yet:
`pdb_id` is a provenance label, not a fetchable, coordinate-matching key --
the panel's actual receptor structure is pocket-aligned and doesn't match
what `Protein.from_pdb_id(pdb_id)` downloads from RCSB (confirmed directly:
the two are ~35-55Å apart for the same "pdb_id"). Blocked on DDOS-7481.

Currently no batching is supported for the docking path (unlike
`deeporigin.docking`'s `batchSize`) — it runs as a single job with a fixed
resource/time budget for the entire ligand set.

Check for gaps -- ligands that failed entirely, or specific ligand × target
cells with no docked pose:

In [ ]:
dock_job.get_undocked_ligands()

In [ ]:
dock_job.get_missing_pairs()

## Working with existing runs

Reload a specific run by execution id, or grab the most recently created one
-- `from_last_run()` returns exactly one execution (whichever ran last,
either method), so it's only useful when that's actually the one you want:

In [ ]:
reloaded = SecondaryPharmacology.from_last_run()
# or: SecondaryPharmacology.from_id("<executionId>")

reloaded.sync()
reloaded.get_results()

`get_results()`'s `method` column tells you which path produced a given
DataFrame — useful once you're just looking at a saved/exported result with
no job object around to check `.method` on.

Starting a brand-new session with no `job`/`dock_job` variables in memory?
`SecondaryPharmacology.list()` lists every execution of this tool — both
methods together, since ligand-ml and docking share one `tool_key` — newest
first. Without `project_id`, it returns every execution the caller can see
across the whole org, not just this project's own runs, so pass it:

In [ ]:
recent = SecondaryPharmacology.list(status=["Completed"], project_id=client.project_id)
[(r.id, r.method, r.status) for r in recent[:5]]

Filter by `.method` to find the specific pair you want -- e.g. the most
recent ligand-ml and docking run of each:

In [ ]:
ml_runs = [r for r in recent if r.method == "ligand-ml"]
dock_runs = [r for r in recent if r.method == "docking"]
ml_runs[0].id, dock_runs[0].id

`from_dto`/`from_id`/`from_last_run` restore `method`, `ligands`, `uniprots`,
`effort`, and `self_test` from the stored execution inputs. `uniprots` of a
loaded run cannot be changed — call `duplicate()` first to get an editable
copy, validated against the current panel.

In [ ]:
editable = reloaded.duplicate()
editable.uniprots = ["P00533"]

### Compare ligand-ML vs docking

`SecondaryPharmacology.plot_ml_vs_docking()` is a `staticmethod` -- it takes
two runs, so it doesn't belong to either one, but it's meaningless without
this class, so it's not a free top-level function either. Reload both by id
rather than resubmitting -- docking especially is real compute, and the
whole point of reloading is not paying for it twice. Using `job.id`/
`dock_job.id` here since they're already in memory from earlier in this
notebook; in a fresh session, use `ml_runs[0].id`/`dock_runs[0].id` from the
`.list()` lookup above instead:

In [ ]:
reloaded_ml = SecondaryPharmacology.from_id(ml_runs[0].id)
reloaded_dock = SecondaryPharmacology.from_id(dock_runs[0].id)
SecondaryPharmacology.plot_ml_vs_docking(reloaded_ml, reloaded_dock)

Each cell splits diagonally: upper-left = `p_active` (ligand-ml), lower-right
= `pose_score` (docking). Same white-to-red scale on both halves, fixed 0-1,
so they're directly comparable at a glance. Rows and columns are the union of
whatever either run covered -- a target only ligand-ml touched renders grey
on the docking half. Sorted so the strongest dual-agreement cells (both
methods say "hit") land top-left.

Reloaded ligands lose their friendly `name` -- execution inputs only store
`id`/`smiles`, not the name -- so labels fall back to a short id suffix
instead of e.g. "gefitinib". Still stable and unique, just less readable.

## Mixing up the two paths

Calling the wrong entry point for `method` raises immediately, telling you
which one to use instead:

In [ ]:
try:
    dock_job.run()  # dock_job is method="docking" -- wrong call
except ValueError as e:
    print(e)